In [ ]:
import os
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine

# Credenciais do seu banco local

usuario_db = 'seu_usuario'           
senha_db = 'sua_senha'               
host_db = 'seu_host'                 
porta_db = 'sua_porta'               
nome_banco_db = 'seu_nome_do_banco' 

DATABASE_URL = f'postgresql://{usuario_db}:{senha_db}@{host_db}:{porta_db}/{nome_banco_db}?client_encoding=utf8'
engine = create_engine(DATABASE_URL)

In [2]:
# Caminho dos arquivos originais
arquivos_path = Path("..") / "Datas"
arquivos_name_list = sorted([arquivo.name for arquivo in arquivos_path.glob("*.csv")])

print("Carregando dados brutos para a tabela de Staging no Postgres...")

# Lendo e enviando bloco por bloco para o banco
for arquivo_name in arquivos_name_list:
    path = arquivos_path / arquivo_name
    # Lemos como string para não quebrar nada na carga bruta
    tdf = pd.read_csv(path, sep=";", encoding="cp1252", dtype=str)
    tdf["arquivo_origem"] = arquivo_name
    
    # Envia para uma tabela chamada 'stg_samp_bruto'
    # 'append' garante que os dados dos 3 anos fiquem na mesma tabela
    tdf.to_sql('stg_samp_bruto', con=engine, if_exists='append', index=False)
    print(f"Arquivo {arquivo_name} carregado na Staging!")

print("Carga bruta inicial concluída com sucesso!")

Carregando dados brutos para a tabela de Staging no Postgres...
Arquivo samp-2024.csv carregado na Staging!
Arquivo samp-2025.csv carregado na Staging!
Arquivo samp-2026.csv carregado na Staging!
Carga bruta inicial concluída com sucesso!


In [4]:
from sqlalchemy import text  # <--- IMPORTANTE: Adicione essa linha aqui

query_dim_tempo = """
CREATE TABLE IF NOT EXISTS dim_tempo_elt (
    tempo_id SERIAL PRIMARY KEY,
    data_competencia DATE UNIQUE,
    ano INT,
    mes INT,
    trimestre INT
);

INSERT INTO dim_tempo_elt (data_competencia, ano, mes, trimestre)
SELECT DISTINCT 
    TO_DATE("DatCompetencia", 'YYYY-MM-DD') as data_competencia,
    EXTRACT(YEAR FROM TO_DATE("DatCompetencia", 'YYYY-MM-DD'))::INT as ano,
    EXTRACT(MONTH FROM TO_DATE("DatCompetencia", 'YYYY-MM-DD'))::INT as mes,
    EXTRACT(QUARTER FROM TO_DATE("DatCompetencia", 'YYYY-MM-DD'))::INT as trimestre
FROM stg_samp_bruto
WHERE "DatCompetencia" IS NOT NULL
ON CONFLICT (data_competencia) DO NOTHING;
"""

with engine.connect() as conexao:
    # Colocamos o text(query_dim_tempo) aqui embaixo para o SQLAlchemy aceitar
    conexao.execute(text(query_dim_tempo))
    conexao.commit()  # Garante que as alterações sejam salvas no Postgres
    print("Dimensão Tempo criada e transformada via SQL com sucesso!")

Dimensão Tempo criada e transformada via SQL com sucesso!


In [5]:
query_outras_dimensoes = """
-- 1. DIMENSÃO DISTRIBUIDORA
CREATE TABLE IF NOT EXISTS dim_distribuidora_elt (
    distribuidora_id SERIAL PRIMARY KEY,
    cnpj_distribuidora VARCHAR(20),
    sigla_distribuidora VARCHAR(50),
    nome_distribuidora VARCHAR(255),
    CONSTRAINT unique_distribuidora UNIQUE (cnpj_distribuidora, sigla_distribuidora, nome_distribuidora)
);

INSERT INTO dim_distribuidora_elt (cnpj_distribuidora, sigla_distribuidora, nome_distribuidora)
SELECT DISTINCT 
    REGEXP_REPLACE("NumCNPJAgenteDistribuidora", '\D', '', 'g') as cnpj_distribuidora,
    TRIM(UPPER("SigAgenteDistribuidora")) as sigla_distribuidora,
    TRIM(UPPER("NomAgenteDistribuidora")) as nome_distribuidora
FROM stg_samp_bruto
WHERE "NumCNPJAgenteDistribuidora" IS NOT NULL
ON CONFLICT (cnpj_distribuidora, sigla_distribuidora, nome_distribuidora) DO NOTHING;


-- 2. DIMENSÃO ACESSANTE
CREATE TABLE IF NOT EXISTS dim_acessante_elt (
    acessante_id SERIAL PRIMARY KEY,
    cnpj_acessante VARCHAR(20),
    ide_acessante VARCHAR(100),
    nome_acessante VARCHAR(255),
    CONSTRAINT unique_acessante UNIQUE (cnpj_acessante, ide_acessante, nome_acessante)
);

INSERT INTO dim_acessante_elt (cnpj_acessante, ide_acessante, nome_acessante)
SELECT DISTINCT 
    COALESCE(REGEXP_REPLACE("NumCNPJAgenteAcessante", '\D', '', 'g'), '00000000000000') as cnpj_acessante,
    COALESCE(TRIM(UPPER("IdeAgenteAcessante")), 'NAO INFORMADO') as ide_acessante,
    COALESCE(TRIM(UPPER("NomAgenteAcessante")), 'NAO INFORMADO') as nome_acessante
FROM stg_samp_bruto
ON CONFLICT (cnpj_acessante, ide_acessante, nome_acessante) DO NOTHING;


-- 3. DIMENSÃO MERCADO
CREATE TABLE IF NOT EXISTS dim_mercado_elt (
    mercado_id SERIAL PRIMARY KEY,
    tipo_mercado VARCHAR(150),
    modalidade_tarifaria VARCHAR(150),
    subgrupo_tarifario VARCHAR(150),
    classe_consumo VARCHAR(150),
    subclasse_consumidor VARCHAR(150),
    detalhe_consumidor VARCHAR(150),
    posto_tarifario VARCHAR(150),
    opcao_energia VARCHAR(150),
    detalhe_mercado VARCHAR(150),
    CONSTRAINT unique_mercado UNIQUE (tipo_mercado, modalidade_tarifaria, subgrupo_tarifario, classe_consumo, subclasse_consumidor, detalhe_consumidor, posto_tarifario, opcao_energia, detalhe_mercado)
);

INSERT INTO dim_mercado_elt (tipo_mercado, modalidade_tarifaria, subgrupo_tarifario, classe_consumo, subclasse_consumidor, detalhe_consumidor, posto_tarifario, opcao_energia, detalhe_mercado)
SELECT DISTINCT 
    TRIM(UPPER("NomTipoMercado")),
    TRIM(UPPER("DscModalidadeTarifaria")),
    TRIM(UPPER("DscSubGrupoTarifario")),
    TRIM(UPPER("DscClasseConsumoMercado")),
    TRIM(UPPER("DscSubClasseConsumidor")),
    TRIM(UPPER("DscDetalheConsumidor")),
    TRIM(UPPER("DscPostoTarifario")),
    TRIM(UPPER("DscOpcaoEnergia")),
    TRIM(UPPER("DscDetalheMercado"))
FROM stg_samp_bruto
ON CONFLICT (tipo_mercado, modalidade_tarifaria, subgrupo_tarifario, classe_consumo, subclasse_consumidor, detalhe_consumidor, posto_tarifario, opcao_energia, detalhe_mercado) DO NOTHING;
"""

with engine.connect() as conexao:
    conexao.execute(text(query_outras_dimensoes))
    conexao.commit()
    print("Dimensões Distribuidora, Acessante e Mercado criadas com SQL!")

<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
C:\Users\souvi\AppData\Local\Temp\ipykernel_55736\2462431618.py:1: SyntaxWarning: invalid escape sequence '\D'
  query_outras_dimensoes = """


Dimensões Distribuidora, Acessante e Mercado criadas com SQL!


In [7]:
query_tabela_fato = """
CREATE TABLE IF NOT EXISTS fato_energia_elt (
    fato_id SERIAL PRIMARY KEY,
    tempo_id INT,
    distribuidora_id INT,
    acessante_id INT,
    mercado_id INT,
    data_geracao DATE,
    valor_mercado NUMERIC(18,2),
    arquivo_origem VARCHAR(100)
);

TRUNCATE TABLE fato_energia_elt;

INSERT INTO fato_energia_elt (tempo_id, distribuidora_id, acessante_id, mercado_id, data_geracao, valor_mercado, arquivo_origem)
SELECT 
    t.tempo_id,
    d.distribuidora_id,
    a.acessante_id,
    m.mercado_id,
    TO_DATE(s."DatGeracaoConjuntoDados", 'YYYY-MM-DD'),
    COALESCE(REPLACE(REPLACE(s."VlrMercado", '.', ''), ',', '.')::NUMERIC, 0),
    s.arquivo_origem
FROM stg_samp_bruto s
LEFT JOIN dim_tempo_elt t ON TO_DATE(s."DatCompetencia", 'YYYY-MM-DD') = t.data_competencia
LEFT JOIN dim_distribuidora_elt d ON REGEXP_REPLACE(s."NumCNPJAgenteDistribuidora", '\D', '', 'g') = d.cnpj_distribuidora 
    AND TRIM(UPPER(s."SigAgenteDistribuidora")) = d.sigla_distribuidora 
    AND TRIM(UPPER(s."NomAgenteDistribuidora")) = d.nome_distribuidora
LEFT JOIN dim_acessante_elt a ON COALESCE(REGEXP_REPLACE(s."NumCNPJAgenteAcessante", '\D', '', 'g'), '00000000000000') = a.cnpj_acessante
    AND COALESCE(TRIM(UPPER(s."IdeAgenteAcessante")), 'NAO INFORMADO') = a.ide_acessante
    AND COALESCE(TRIM(UPPER(s."NomAgenteAcessante")), 'NAO INFORMADO') = a.nome_acessante
LEFT JOIN dim_mercado_elt m ON TRIM(UPPER(s."NomTipoMercado")) = m.tipo_mercado
    AND TRIM(UPPER(s."DscModalidadeTarifaria")) = m.modalidade_tarifaria
    AND TRIM(UPPER(s."DscSubGrupoTarifario")) = m.subgrupo_tarifario
    AND TRIM(UPPER(s."DscClasseConsumoMercado")) = m.classe_consumo
    AND TRIM(UPPER(s."DscSubClasseConsumidor")) = m.subclasse_consumidor
    AND TRIM(UPPER(s."DscDetalheConsumidor")) = m.detalhe_consumidor
    AND TRIM(UPPER(s."DscPostoTarifario")) = m.posto_tarifario
    AND TRIM(UPPER(s."DscOpcaoEnergia")) = m.opcao_energia
    AND TRIM(UPPER(s."DscDetalheMercado")) = m.detalhe_mercado
WHERE s."DatCompetencia" IS NOT NULL AND s."VlrMercado" IS NOT NULL;
"""

print("Criando e povoando a Tabela Fato via SQL...")
with engine.connect() as conexao:
    conexao.execute(text(query_tabela_fato))
    conexao.commit()
    print("PROCESSO DE ELT CONCLUÍDO COM SUCESSO NO POSTGRESQL!")

<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
C:\Users\souvi\AppData\Local\Temp\ipykernel_55736\2852467889.py:1: SyntaxWarning: invalid escape sequence '\D'
  query_tabela_fato = """


Criando e povoando a Tabela Fato via SQL...
PROCESSO DE ELT CONCLUÍDO COM SUCESSO NO POSTGRESQL!
